# Lesson 12b: Fine-tuning and Adaptation — Practical

12a derived, in NumPy, why full fine-tuning, feature extraction and prompting
trade cost for accuracy, why LoRA's low-rank factorisation can match full
fine-tuning at a fraction of the trainable parameters, and why a smaller
update does not automatically forget less. This notebook reproduces the
prompting-vs-fine-tuning-vs-LoRA comparison with the actual production tools —
a real pretrained encoder, a real sentiment dataset, Hugging Face `transformers`
for full fine-tuning, and `peft` for LoRA — to see whether 12a's story survives
contact with a real model.

## Introduction

The encoder is `google/bert_uncased_L-2_H-128_A-2` ("BERT-tiny"), the same
small pretrained model 8b used to inspect real embeddings — two transformer
layers, 128 hidden units, about 4.4 million parameters, small enough to
fine-tune on CPU in seconds. The task is binary sentiment classification on a
subset of SST-2 (Stanford Sentiment Treebank), via the Hugging Face `datasets`
library.

Three things are compared on the same held-out test examples: a **zero-shot**
baseline that never trains on this task at all (prompting BERT-tiny's own
masked-language-modelling head), **full fine-tuning** of every parameter with
`transformers`, and **LoRA** adapters trained with `peft` at several ranks.

## Setup

In [ ]:
# Colab does not ship `peft` by default; install it only if it is missing so a
# local run (where requirements.txt already installed it) stays fast.
try:
    import peft  # noqa: F401
except ImportError:
    %pip install -q "peft>=0.10.0"

import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)

import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

plt.rcParams["figure.figsize"] = (6, 4)
print("torch:", torch.__version__)

MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"

In [ ]:
# A small, fixed subset of SST-2 keeps every run here well under a minute.
N_TRAIN, N_TEST = 300, 200

sst2_train = load_dataset("SetFit/sst2", split="train")
sst2_test = load_dataset("SetFit/sst2", split="validation")

rng = np.random.default_rng(SEED)
train_idx = rng.choice(len(sst2_train), size=N_TRAIN, replace=False)
test_idx = rng.choice(len(sst2_test), size=N_TEST, replace=False)

train_texts = [sst2_train[int(i)]["text"] for i in train_idx]
train_labels = torch.tensor([sst2_train[int(i)]["label"] for i in train_idx])
test_texts = [sst2_test[int(i)]["text"] for i in test_idx]
test_labels = torch.tensor([sst2_test[int(i)]["label"] for i in test_idx])

print(f"train: {len(train_texts)} examples, {train_labels.float().mean():.2f} positive")
print(f"test:  {len(test_texts)} examples, {test_labels.float().mean():.2f} positive")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_enc = tokenizer(train_texts, padding=True, truncation=True, max_length=48, return_tensors="pt")
test_enc = tokenizer(test_texts, padding=True, truncation=True, max_length=48, return_tensors="pt")

## Fine-Tuning a Small Encoder

**Zero-shot / prompting baseline first**, exactly as 12a's *Adaptation
Strategies* section defined it: use the pretrained model exactly as it is, no
parameters touched. BERT-tiny was pretrained as a masked language model, so
the natural prompt is a cloze test: append `"It was [MASK]."` to a review and
ask the model's own MLM head whether *good* or *bad* is the more probable
completion — no task-specific training, no labelled examples, just the
knowledge already in the pretrained weights.

In [ ]:
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.eval()

pos_id = tokenizer.convert_tokens_to_ids("good")
neg_id = tokenizer.convert_tokens_to_ids("bad")


def zero_shot_predict(texts):
    preds = []
    with torch.no_grad():
        for t in texts:
            prompt = f"{t} It was {tokenizer.mask_token}."
            enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=64)
            mask_pos = (enc["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0][0]
            logits = mlm_model(**enc).logits[0, mask_pos]
            preds.append(1 if logits[pos_id] > logits[neg_id] else 0)
    return torch.tensor(preds)


zero_shot_preds = zero_shot_predict(test_texts)
acc_zero_shot = (zero_shot_preds == test_labels).float().mean().item()
print(f"zero-shot ('It was good' vs 'It was bad') accuracy: {acc_zero_shot:.3f}")

Now full fine-tuning: every one of BERT-tiny's ~4.4M parameters, including a
freshly-initialised classification head, updated by gradient descent on the
300 labelled training examples. A full fine-tune needs a much smaller learning
rate than the LoRA runs below — with every parameter free to move, a large
step risks wrecking the pretrained representations before the classification
head has learned anything useful from them.

In [ ]:
def evaluate(model, enc, labels, batch_size=32):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(labels), batch_size):
            batch = {k: v[i:i + batch_size] for k, v in enc.items()}
            preds.append(model(**batch).logits.argmax(dim=-1))
    return (torch.cat(preds) == labels).float().mean().item()


def train_classifier(model, enc, labels, epochs, lr, batch_size=16, seed=SEED):
    torch.manual_seed(seed)
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    n = len(labels)
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            batch = {k: v[idx] for k, v in enc.items()}
            loss = model(**batch, labels=labels[idx]).loss
            loss.backward()
            opt.step()
            opt.zero_grad()
    return model


torch.manual_seed(SEED)
full_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
n_full_params = sum(p.numel() for p in full_model.parameters())

acc_untrained_head = evaluate(full_model, test_enc, test_labels)
full_model = train_classifier(full_model, train_enc, train_labels, epochs=10, lr=1e-4)
acc_full_finetune = evaluate(full_model, test_enc, test_labels)

print(f"zero-shot (MLM prompt):        {acc_zero_shot:.3f}")
print(f"untrained classification head: {acc_untrained_head:.3f}  (chance -- head is random)")
print(f"full fine-tune (all {n_full_params:,} params): {acc_full_finetune:.3f}")

## LoRA with PEFT

12a derived $\Delta W = BA$ and trained it by hand; `peft`'s `LoraConfig`
does the identical factorisation, wired automatically into whichever weight
matrices are named in `target_modules` -- here, BERT's attention query and
value projections. LoRA's own parameters ($B$, zero-initialised, and $A$)
start far smaller in scale than the frozen pretrained weights they are added
to, so they need a *larger* learning rate than full fine-tuning to move
enough in the same number of steps -- the opposite tuning direction from the
cell above, for exactly the reason 12a's derivation predicts: a rank-$r$
factorisation has far fewer degrees of freedom to spread a given amount of
learning across.

In [ ]:
def make_lora_model(rank):
    torch.manual_seed(SEED)
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=rank, lora_alpha=16, lora_dropout=0.05,
        target_modules=["query", "value"],
    )
    return get_peft_model(base, cfg)


rank_default = 4
lora_model = make_lora_model(rank_default)
n_lora_trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
lora_model.print_trainable_parameters()

lora_model = train_classifier(lora_model, train_enc, train_labels, epochs=10, lr=2e-3)
acc_lora_default = evaluate(lora_model, test_enc, test_labels)

print(f"\nLoRA (rank={rank_default}): {n_lora_trainable:,} trainable params "
      f"({100 * n_lora_trainable / n_full_params:.2f}% of full fine-tuning)")
print(f"LoRA (rank={rank_default}) test accuracy: {acc_lora_default:.3f}")

Sweep the rank to see 12a's "even a small rank recovers most of the
accuracy" prediction hold (or not) against a real model and real data.

In [ ]:
ranks = [1, 2, 4, 8, 16]
lora_trainable_params, lora_accs = [], []

for r in ranks:
    m = make_lora_model(r)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    m = train_classifier(m, train_enc, train_labels, epochs=10, lr=2e-3)
    acc = evaluate(m, test_enc, test_labels)
    lora_trainable_params.append(trainable)
    lora_accs.append(acc)
    print(f"rank={r:>2}  trainable={trainable:>6} ({100 * trainable / n_full_params:5.2f}% of full)  "
          f"test accuracy={acc:.3f}")

plt.figure()
plt.plot(ranks, lora_accs, "o-", label="LoRA")
plt.axhline(acc_full_finetune, color="gray", linestyle="--", label="full fine-tuning")
plt.axhline(acc_zero_shot, color="black", linestyle=":", label="zero-shot")
plt.xlabel("LoRA rank r")
plt.ylabel("test accuracy")
plt.title("LoRA rank vs. accuracy on real BERT-tiny + SST-2")
plt.legend()
plt.tight_layout()
plt.show()

## Comparison

In [ ]:
methods = ["zero-shot\n(MLM prompt)", "untrained head\n(chance)",
           f"LoRA r={rank_default}", "full fine-tune"]
accs = [acc_zero_shot, acc_untrained_head, acc_lora_default, acc_full_finetune]
params = [0, 0, n_lora_trainable, n_full_params]

for m, a, p in zip(methods, accs, params):
    print(f"{m.replace(chr(10), ' '):<24} accuracy={a:.3f}  trainable_params={p:,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(methods, accs, color=["black", "gray", "tab:orange", "tab:red"])
ax1.set_ylabel("test accuracy")
ax1.set_title("Accuracy by method")
ax1.set_ylim(0, 1)

ax2.plot(ranks, lora_trainable_params, "o-", label="LoRA")
ax2.axhline(n_full_params, color="gray", linestyle="--", label="full fine-tuning")
ax2.set_yscale("log")
ax2.set_xlabel("LoRA rank r")
ax2.set_ylabel("trainable parameters (log scale)")
ax2.set_title("Parameter cost by rank")
ax2.legend()
plt.tight_layout()
plt.show()

Most of 12a's predictions hold up well against the real model — but not all
of them, and the one that does not is worth reporting rather than smoothing
over. Zero-shot prompting scores statistically indistinguishable from the
untrained (chance-level) classification head: both sit within noise of 50%.
Prompting's *ceiling* was always the point 12a made about this strategy, and
here that ceiling turns out to be barely above chance — plausibly because
BERT-tiny is a genuinely tiny, two-layer model, and a single fixed cloze
template ("It was [MASK].") is a narrow way to query whatever sentiment
knowledge two layers actually picked up during pretraining. A larger encoder
or a better-tuned prompt might do better; this one does not, and the honest
comparison shows that plainly rather than asserting it should.

Full fine-tuning and LoRA, in contrast, both work clearly and as predicted:
both push far past either baseline because both actually train on the 300
labelled examples, rather than only querying pretrained knowledge. LoRA
reaches accuracy close to full fine-tuning's while training well under one
percent of the parameters — the low-rank hypothesis 12a derived in NumPy
holds just as well for a real pretrained transformer's attention weights.
The rank sweep is close to flat across the ranks tried here: on 300 labelled
examples, there is not enough signal for a larger rank to buy much extra
accuracy, echoing 12a's finding that even a small rank can already capture
what a small fine-tuning task needs.

## Key Takeaways

- **The same three-way comparison from 12a reproduces on a real model**:
  zero-shot prompting (BERT-tiny's own masked-language-modelling head, no
  training), full fine-tuning (every parameter, `transformers`), and LoRA
  (`peft`, a small added subspace) trade cost for accuracy in the predicted
  order.
- **LoRA and full fine-tuning need different learning rates**: full
  fine-tuning moves every pretrained weight and needs a small step size to
  avoid wrecking them; LoRA's added parameters start at a much smaller
  effective scale and need a larger step size to move enough in the same
  number of steps.
- **LoRA recovers most of full fine-tuning's accuracy at well under 1% of the
  trainable parameters** on this task, and the accuracy is close to flat
  across the ranks tried — consistent with 12a's NumPy finding that a small
  rank is often already enough.
- **Prompting's ceiling is real, and it can sit barely above chance**: on a
  genuinely tiny two-layer encoder with a single fixed cloze template,
  zero-shot accuracy here was statistically indistinguishable from an
  untrained (chance-level) classification head — a result worth reporting
  plainly, not a failure to explain away, since it is exactly the capped
  accuracy 12a's theory predicted for prompting, just capped lower than one
  might have hoped.